# MELD multimodal emotion classifier (Text Ablation)
Use Python 3.10+ and install `requirements.txt` in the selected kernel.
Training requires MELD annotations, extracted video clips, and 16 kHz mono WAV files.
Run `python extract_audio.py` after extracting videos (see README.md). Configure `data_dir` for your installation.


In [ ]:
%pip install -r requirements.txt
# Transformers 4 VideoMAE uses Pillow/NumPy; torchvision is optional.
# Remove it to avoid incompatible native operators (torchvision::nms).
%pip uninstall -y torchvision
# REQUIRED: restart the kernel after this cell, then run the remaining cells.


In [ ]:
# This notebook uses PyTorch only. Set before importing transformers.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

# Run after installing requirements and restarting the kernel.
import sys
from importlib.metadata import PackageNotFoundError, version

print("Notebook Python:", sys.executable)
for package in ("torch", "transformers", "sentencepiece", "protobuf", "ipywidgets"):
    try:
        installed = version(package)
    except PackageNotFoundError:
        raise RuntimeError(
            f"Missing {package}. Run the %pip install cell, restart the kernel, and retry."
        ) from None
    print(f"{package}: {installed}")
    if package == "transformers" and installed.split(".")[0] != "4":
        raise RuntimeError(
            "This notebook requires transformers>=4.40,<5. "
            "Run %pip install -r requirements.txt and restart the kernel."
        )


In [ ]:
import numpy as np
import torch
import torch.nn as nn


In [ ]:
import random
import numpy as np

SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

In [ ]:
import pandas as pd
import os

data_dir = "datasets/MELD"

train = pd.read_csv(os.path.join(data_dir, "train_sent_emo.csv"))
# print(train["Utterance"][0])
train


In [ ]:
from torch.utils.data import Dataset, DataLoader
import os



class MELDDataset(Dataset):
    def __init__(self, csv_path, mode):
        df = pd.read_csv(csv_path)
        self.data = []
        excluded = {"train": {(125, 3)}, "dev": {(110, 7)}}.get(mode, set())
        
        for i in range(len(df)):
            if (df["Dialogue_ID"][i], df["Utterance_ID"][i]) in excluded:
                continue
            utterance = df["Utterance"][i]
            emotion = df["Emotion"][i]
            to_append = {
                "text": utterance,
                "label": emotion,
            }
            self.data.append(to_append)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, key):
        return self.data[key]

train_data_path = os.path.join(data_dir, "train_sent_emo.csv")
dev_data_path = os.path.join(data_dir, "dev_sent_emo.csv")
test_data_path = os.path.join(data_dir, "test_sent_emo.csv")


# Point these folders at the extracted MELD clips if their names differ.


train_dataset = MELDDataset(train_data_path, mode="train")
dev_dataset = MELDDataset(dev_data_path, mode="dev")
test_dataset = MELDDataset(test_data_path, mode="test")

train_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=2, shuffle=False)
test_data_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# This notebook uses PyTorch only. Set before importing transformers.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import cv2
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoModel,
    AutoTokenizer,
)

import sentencepiece
import librosa

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=False)


emotions_to_id = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "joy": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}




device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

class MultimodalModel(nn.Module):
    def __init__(self, text_model):
        super().__init__()
        self.text_model = text_model


        # Only train the fusion classifier.
        for param in self.text_model.parameters():
            param.requires_grad = False
 

        self.classifier = nn.Sequential(
            nn.Linear(
                text_model.config.hidden_size,
                512,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, len(emotions_to_id)),
        )

    def train(self, mode=True):
        super().train(mode)
        # Frozen encoders should remain deterministic; classifier dropout still trains.
        self.text_model.eval()

        return self

    def forward(self, text):
        device = next(self.classifier.parameters()).device
        text_tokens = tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)
        

 
        


        with torch.no_grad():
            text_hidden = self.text_model(**text_tokens).last_hidden_state
            text_mask = text_tokens["attention_mask"].unsqueeze(-1).to(text_hidden.dtype)



            text = (text_hidden * text_mask).sum(dim=1) / text_mask.sum(dim=1).clamp(min=1)
      

        return self.classifier(text)



In [ ]:
text_model = AutoModel.from_pretrained(
    "microsoft/deberta-v3-small"
)

seed_everything(SEED)

model = MultimodalModel(text_model=text_model)

model = model.to(device)

num_epochs = 10
learning_rate = 5e-4

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=learning_rate)

# This is the loss fn for the unweight model. Uncomment to reproduce mel_classfier_baseline.pt
criterion = nn.CrossEntropyLoss()

# producing meld_classifier_audio.pt
# train_label_ids = torch.tensor([emotions_to_id[sample["label"]] for sample in train_dataset])
# class_counts = torch.bincount(train_label_ids, minlength = len(emotions_to_id))
# balanced_weights = (
#     len(train_label_ids)
#     / (len(emotions_to_id) * class_counts)
# )
# class_weights = torch.sqrt(balanced_weights).to(device)


# print({
#     emotion: round(class_weights[label_id].item(), 3)
#     for emotion, label_id in emotions_to_id.items()
# })

# criterion = nn.CrossEntropyLoss(weight=class_weights)


def batch_labels(batch):
    return torch.tensor([emotions_to_id[label] for label in batch["label"]], dtype=torch.long, device=device)


@torch.no_grad()
def accuracy(loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        # Convert dataset labels to numeric IDs.
        labels = batch_labels(batch)
        # Text and video are preprocessed inside the model.
        predictions = model(batch["text"]).argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return correct / total


In [ ]:
seed_everything(SEED)
train_generator.manual_seed(SEED)

for epoch in range(num_epochs):
    model.train()
    total_loss, total_examples = 0.0, 0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(batch["text"])
        # Convert dataset labels to numeric IDs.
        labels = batch_labels(batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    print(f"Epoch {epoch + 1}/{num_epochs} | loss: {total_loss / total_examples:.4f} | "
          f"validation accuracy: {accuracy(dev_loader):.2%}")


print(f"Test accuracy: {accuracy(test_data_loader):.2%}")
# Only the classifier learns; reuse the same pretrained encoders when loading it.
torch.save({"classifier": model.classifier.state_dict(), "labels": emotions_to_id}, "meld_text_only.pt")


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

checkpoint = torch.load("meld_text_only.pt", map_location=device, weights_only=True)
model = MultimodalModel(text_model=text_model).to(device)
model.classifier.load_state_dict(checkpoint["classifier"])
model.eval()

import time

all_labels = []
all_predictions = []
latency_times = []

with torch.no_grad():
    for batch in test_data_loader:
        start_time = time.time()
        labels = batch_labels(batch)
        predictions = model(batch["text"]).argmax(dim=-1)
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        end_time = time.time()
        latency = end_time - start_time
        latency_times.append(latency)

label_ids = list(range(len(emotions_to_id)))
emotion_names = [
    emotion
    for emotion, label_id in sorted(emotions_to_id.items(), key=lambda item: item[1])
]

print(f"Accuracy: {accuracy_score(all_labels, all_predictions):.2%}")
print(f"Weighted F1: {f1_score(all_labels, all_predictions, average='weighted', zero_division=0):.2%}")
print(f"Macro F1: {f1_score(all_labels, all_predictions, average='macro', zero_division=0):.2%}")
print("\nPer-class metrics:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=label_ids,
    target_names=emotion_names,
    digits=4,
    zero_division=0,
))
print("latency: ", latency_times)
print("average latency: ", sum(latency_times)/len(latency_times))


# Audio-model prediction
For prediction only, install dependencies and restart the kernel. Run the environment-check cell,
the cell beginning `from concurrent.futures import ThreadPoolExecutor`, and the setup cell below.
The setup loads `meld_classifier_audio.pt` and all three pretrained encoders.
Call `predict_emotion(transcript, video_path, audio_path)` with matching text, video, and WAV audio.
The example using `test_dataset[0]` also requires the dataset cells and local MELD media.
This model does not transcribe speech. Softmax scores are not calibrated certainty.
For the text-and-video widget using the baseline checkpoint, open `demo.ipynb`.


In [ ]:
from pathlib import Path

if "MultimodalModel" not in globals():
    raise RuntimeError("Run the tokenizer / video reader / MultimodalModel definition cell first.")

checkpoint_path = Path("meld_text_only.pt").resolve()


checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

inference_device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

text_model = AutoModel.from_pretrained("microsoft/deberta-v3-small")

inference_model = MultimodalModel(text_model=text_model)
inference_model.classifier.load_state_dict(checkpoint["classifier"])
inference_model = inference_model.to(inference_device).eval()

import json
import random

with open("responses.json", 'r', encoding='utf-8') as file:
    data = json.load(file)
responses = data["responses"]



import time

@torch.inference_mode()
def predict_emotion(transcript):
    transcript = transcript.strip()
    if not transcript:
        raise ValueError("Enter the words spoken in the clip.")


    inference_model.eval()

    start_time = time.time()
    # put in data as ([a tuple of list]) -> ([transcript], [video], [etc])
    logits = inference_model([transcript])
    probabilities = logits.softmax(dim=-1)[0].cpu().tolist()

    end_time = time.time()

    list_of_prob = sorted(
        [(emotion, probabilities[label_id]) for emotion, label_id in emotions_to_id.items()],
        key=lambda item: item[1],
        reverse=True,
    )
    random_index = random.randint(0,2)
    emotion = list_of_prob[0][0]
    confidence = list_of_prob[0][1]
    response = responses[emotion][random_index]
    latency = end_time - start_time
    
    if confidence > 0.4:
        return {
            "emotion": emotion,
            "confidence": confidence,
            "response": response,
            "latency": latency
        }
    else:
        return "No emotion confidence exceeds 0.4", list_of_prob


print(f"Ready on {inference_device}. Relative video paths start at: {Path.cwd()}")


In [ ]:
sample = test_dataset[0]
print(predict_emotion(sample["text"]))